# Task 01 — DataFrame Basics

Create DataFrames, select, filter, add columns, handle nulls.

## Setup

In [ ]:
import os, shutil, subprocess, sys

def _find_java():
    """Check if java is available on PATH or in JAVA_HOME."""
    if os.environ.get("JAVA_HOME"):
        java_bin = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
        if os.path.isfile(java_bin):
            return True
    return shutil.which("java") is not None

def _find_installed_jdk():
    """Look for a previously installed JDK in ~/.jdk."""
    jdk_dir = os.path.expanduser("~/.jdk")
    if os.path.exists(jdk_dir):
        for d in sorted(os.listdir(jdk_dir)):
            candidate = os.path.join(jdk_dir, d)
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "bin", "java")):
                return candidate
    return None

# Auto-install Java if not available (required by PySpark)
if not _find_java():
    prev = _find_installed_jdk()
    if prev:
        os.environ["JAVA_HOME"] = prev
        print(f"Using JAVA_HOME={prev}")
    else:
        print("Java not found. Installing JDK 17 via install-jdk...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "install-jdk"])
        import jdk
        path = jdk.install("17")
        os.environ["JAVA_HOME"] = path
        print(f"JAVA_HOME set to {path}")
else:
    print(f"Java found. JAVA_HOME={os.environ.get('JAVA_HOME', '(system)')}")

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Task01") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

print("Setup complete.")

## Task 1.1: Create a DataFrame from a list

Create a DataFrame `products_df` from the following data with columns: `name` (string), `category` (string), `price` (double).

Data:
- ("Laptop", "Electronics", 1200.0)
- ("Desk", "Furniture", 350.0)
- ("Phone", "Electronics", 800.0)
- ("Chair", "Furniture", 250.0)
- ("Tablet", "Electronics", 500.0)

In [ ]:
# YOUR CODE HERE
products_df = ...

# TEST — Do not modify
assert products_df.count() == 5, f"Expected 5 rows, got {products_df.count()}"
assert set(products_df.columns) == {"name", "category", "price"}
assert products_df.schema["price"].dataType == DoubleType()
rows = {r["name"]: r["price"] for r in products_df.collect()}
assert rows["Laptop"] == 1200.0
print("Task 1.1 passed!")

## Task 1.2: Read CSV and basic stats

Read `employees.csv` into `emp_df`. Then create `eng_df` containing only Engineering department employees with salary > 95000, sorted by salary descending.

In [ ]:
# YOUR CODE HERE
emp_df = ...
eng_df = ...

# TEST — Do not modify
assert emp_df.count() == 15, f"Expected 15, got {emp_df.count()}"
assert eng_df.count() == 3, f"Expected 3 high-paid engineers, got {eng_df.count()}"
salaries = [r["salary"] for r in eng_df.collect()]
assert salaries == sorted(salaries, reverse=True), "Should be sorted desc"
assert all(s > 95000 for s in salaries)
print("Task 1.2 passed!")

## Task 1.3: Add computed columns

Starting from `emp_df` (read employees.csv again if needed), create `emp_enhanced` with two new columns:
- `salary_band`: "Senior" if salary >= 95000, "Mid" if >= 75000, else "Junior"
- `tenure_years`: number of full years from `hire_date` to 2024-01-01 (use `datediff` and divide by 365, cast to int)

In [ ]:
# YOUR CODE HERE
emp_df = spark.read.csv(os.path.join(FIXTURES, "employees.csv"), header=True, inferSchema=True)
emp_enhanced = ...

# TEST — Do not modify
assert "salary_band" in emp_enhanced.columns
assert "tenure_years" in emp_enhanced.columns
bands = {r["name"]: r["salary_band"] for r in emp_enhanced.collect()}
assert bands["Noah"] == "Senior"   # 105000
assert bands["Charlie"] == "Junior"  # 72000
assert bands["Diana"] == "Mid"    # 78000
years = {r["name"]: r["tenure_years"] for r in emp_enhanced.collect()}
assert years["Noah"] >= 6, f"Noah should have 6+ years, got {years['Noah']}"
assert years["Ivy"] <= 2, f"Ivy should have <=2 years, got {years['Ivy']}"
print("Task 1.3 passed!")

## Task 1.4: Handle nulls

Create a DataFrame `df_nulls` from this data: `[(1, "Alice", 100.0), (2, None, 200.0), (3, "Charlie", None), (4, None, None)]` with columns `id`, `name`, `value`.

Then create:
- `df_filled`: nulls filled — name with "Unknown", value with 0.0
- `df_complete`: only rows where ALL columns are non-null

In [ ]:
# YOUR CODE HERE
df_nulls = ...
df_filled = ...
df_complete = ...

# TEST — Do not modify
assert df_nulls.count() == 4
assert df_filled.filter(F.col("name").isNull()).count() == 0
assert df_filled.filter(F.col("value").isNull()).count() == 0
filled = {r["id"]: (r["name"], r["value"]) for r in df_filled.collect()}
assert filled[2] == ("Unknown", 200.0)
assert filled[4] == ("Unknown", 0.0)
assert df_complete.count() == 1  # only Alice has all values
print("Task 1.4 passed!")

## Task 1.5: String and date functions

From `emp_df`, create `emp_strings` with columns:
- `name_upper`: name in upper case
- `dept_lower`: department in lower case
- `hire_year`: year extracted from hire_date
- `name_length`: length of the name

In [ ]:
# YOUR CODE HERE
emp_strings = ...

# TEST — Do not modify
row = emp_strings.filter(F.col("name") == "Alice").collect()[0]
assert row["name_upper"] == "ALICE"
assert row["dept_lower"] == "engineering"
assert row["hire_year"] == 2020
assert row["name_length"] == 5
print("Task 1.5 passed!")

## Cleanup

In [ ]:
spark.stop()
print("All tasks done!")